# Baseline · Agente investigador sobre informes 10-K

Preparación para el 17 de septiembre. Ejecutad las celdas en orden con Python
3.11 o posterior. Se puede abrir desde la raíz del repositorio o desde `src`.

Las celdas marcadas como **Código de clase** se reutilizan del
notebook de `Material_Clase`. `miax_s1.py` se importa desde esa carpeta, sin
copiarlo ni modificarlo. En los dos ejercicios solo se sustituye el `...`
pendiente. Las celdas **Código añadido** mantienen el estilo del material.
Los metadatos de cada celda original guardan su procedencia y huella.

Este es el punto de partida: conserva la búsqueda, las herramientas y el
agente de clase. Las mejoras de retrieval, el middleware de cifras y los
evaluadores completos corresponden a la siguiente sesión.

Necesitáis el corpus completo y una clave `OPENROUTER_API_KEY`. La clave se
pide sin mostrarla. Las celdas de clase incluyen consultas de demostración
que consumen API; la evaluación se ejecuta solo al activar su última celda.

Ajuste de ejecución: se limita la salida del modelo a 1.024 tokens por
llamada para evitar el error de crédito por reservar 65.536 tokens. El agente
reutiliza ese modelo configurado. Estas dos adaptaciones se documentan en
los metadatos; los archivos de `Material_Clase` permanecen intactos.
El límite es por llamada, no un presupuesto total. Si una respuesta queda
truncada, revisad el límite y el saldo antes de aumentarlo.


## 1. Preparar las rutas y montar el corpus

La siguiente celda prepara las rutas del proyecto. Después se ejecuta
**la celda original de clase, copiada literalmente**, que localiza los dos
ZIP, comprueba sus huellas SHA-256 y los descomprime en `corpus/`.

Dejad `corpus_miax_2026.zip` e `indice_faiss.zip` en la raíz del proyecto,
en `dataset/`, en `Material_Clase/` o junto a este notebook, en `src/`.
La preparación copia los ZIP a la raíz cuando hace falta para que el
código original los encuentre sin cambiar sus rutas.

El material indica que los ZIP los proporciona el profesor por Drive o
el aula virtual. La celda no incluye un enlace de descarga: `URL_RESPALDO`
está vacío. En Colab también permite buscarlos en las rutas de Drive
indicadas en el propio código, una vez montado Drive.

In [15]:
# Código añadido. Preparar las rutas sin modificar la celda de clase.
from pathlib import Path
import hashlib
import os
import shutil
import sys

candidatos = [Path.cwd(), *Path.cwd().parents]
RAIZ = next((p for p in candidatos
             if (p / "Material_Clase/miax_s1.py").is_file()), None)
if RAIZ is None:
    raise FileNotFoundError("Abre este notebook desde el repositorio.")

# Un notebook puede conservar el kernel seleccionado en otro taller.
# La ruta por si sola no demuestra que sus dependencias sean incompatibles.
entorno_esperado = RAIZ / ".venv"
if entorno_esperado.is_dir() and Path(sys.prefix).resolve() != entorno_esperado.resolve():
    print(
        f"AVISO: este notebook usa otro entorno: {sys.executable}. "
        "Para usar las dependencias verificadas, selecciona como kernel "
        f"{entorno_esperado / 'Scripts' / 'python.exe'}. "
        "Si cambias el kernel, ejecuta las celdas desde el principio.")
print("Python del notebook:", sys.executable)

os.chdir(RAIZ)
material = str(RAIZ / "Material_Clase")
if material not in sys.path:
    sys.path.insert(0, material)

for nombre in ["corpus_miax_2026.zip", "indice_faiss.zip"]:
    salida = RAIZ / nombre
    if not salida.is_file():
        bases = [RAIZ / "dataset", RAIZ / "Material_Clase", RAIZ / "src"]
        entrada = next((b / nombre for b in bases
                        if (b / nombre).is_file()), None)
        if entrada is not None:
            shutil.copy2(entrada, salida)
            print("ZIP localizado:", entrada)

print("Proyecto:", RAIZ)

AVISO: este notebook usa otro entorno: c:\Users\Natalia\workplace\Taller-B5-T1-Generativos\.venv\Scripts\python.exe. Para usar las dependencias verificadas, selecciona como kernel c:\Users\Natalia\workplace\Taller-B5-T5-NPL\.venv\Scripts\python.exe. Si cambias el kernel, ejecuta las celdas desde el principio.
Python del notebook: c:\Users\Natalia\workplace\Taller-B5-T1-Generativos\.venv\Scripts\python.exe
Proyecto: c:\Users\Natalia\workplace\Taller-B5-T5-NPL


In [16]:
# %% Corpus e indice  --------------------------------
# Los dos ZIP os los pasamos nosotros (Drive compartido, aula virtual o el
# panel de ficheros de Colab): son 5,6 MB entre los dos. Nada de descargar
# de EDGAR en vivo, que con treinta cuadernos a la vez acaba en bloqueo.
#
# Si los teneis en Drive:
#     from google.colab import drive; drive.mount("/content/drive")
# y anadid la carpeta a CANDIDATOS.
import hashlib, pathlib, zipfile

PAQUETES = [
    ("corpus_miax_2026.zip", "4233c37fc9e9d12091af7a146063ad70903a3fe51404a485854f4021c63daee4"),
    ("indice_faiss.zip", "6b5610ad8ac6ea50364445d39bb464d993cbd87048fb07c4fe16657d7ac11655"),
]
URL_RESPALDO = ""          # vacio si no estan alojados
DESTINO = pathlib.Path("corpus")

CANDIDATOS = [
    pathlib.Path("."),
    pathlib.Path("/content"),
    pathlib.Path("/content/drive/MyDrive/MIAX_2026"),
    pathlib.Path("/content/drive/Shareddrives/MIAX_2026"),
]


def _sha256(ruta):
    d = hashlib.sha256()
    with open(ruta, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            d.update(b)
    return d.hexdigest()


def _localizar(nombre):
    for base in CANDIDATOS:
        ruta = base / nombre
        if ruta.is_file():
            return ruta
    if URL_RESPALDO:
        import urllib.request
        destino = pathlib.Path(nombre)
        urllib.request.urlretrieve(f"{URL_RESPALDO}/{nombre}", destino)
        return destino
    return None


try:
    for nombre, esperado in PAQUETES:
        origen = _localizar(nombre)
        assert origen is not None, (
            f"No encuentro {nombre}. Subelo con el panel de ficheros de "
            f"Colab (icono de carpeta a la izquierda), o monta el Drive "
            f"donde este. Buscado en: {[str(c) for c in CANDIDATOS]}"
        )
        obtenido = _sha256(origen)
        assert obtenido == esperado, (
            f"{nombre} no coincide con lo esperado: el fichero esta "
            f"corrupto o es de otra version.\n"
            f"  esperado: {esperado}\n  obtenido: {obtenido}"
        )
        with zipfile.ZipFile(origen) as zf:
            zf.extractall(DESTINO)

    # Los dos manifiestos declaran el hash de chunks.jsonl. El indice se
    # construyo sobre ESE fichero: si no cuadra, el indice y sus metadatos
    # estan desalineados y el retrieval devuelve el texto equivocado sin
    # dar ningun error.
    huella = _sha256(DESTINO / "chunks.jsonl")
    for manifiesto in ("MANIFEST.md", "indice/MANIFEST.md"):
        ruta = DESTINO / manifiesto
        if ruta.exists():
            assert huella in ruta.read_text(encoding="utf-8"), (
                f"chunks.jsonl no cuadra con {manifiesto}: el indice se "
                "construyo sobre otros fragmentos."
            )

    print("Corpus e indice verificados en", DESTINO.resolve())
    for p in sorted(DESTINO.rglob("*")):
        if p.is_file():
            rel = str(p.relative_to(DESTINO))
            print(f"  {rel:28s} {p.stat().st_size / 1e6:7.2f} MB")

except Exception as e:
    print("No se pudo preparar el corpus:", e)
    print("Pide los ficheros al profesor y dejalos junto al notebook.")

Corpus e indice verificados en C:\Users\Natalia\workplace\Taller-B5-T5-NPL\corpus
  chunks.jsonl                    3.80 MB
  indice\chunks_meta.parquet      1.48 MB
  indice\corpus.faiss             2.69 MB
  indice\MANIFEST.md              0.00 MB
  LEEME.md                        0.00 MB
  MANIFEST.md                     0.00 MB
  secciones.jsonl                 3.21 MB
  xbrl_facts.parquet              0.01 MB


**Código añadido.** Comprobar el resultado antes de instalar o llamar al
modelo. También se admiten los archivos ya descomprimidos en `dataset/`.
En ese caso, la celda de clase puede avisar de que no encuentra los ZIP;
la comprobación siguiente confirma si el corpus completo está disponible.

In [17]:
# El código de clase imprime sus errores. Aquí detenemos la ejecución
# si faltan datos, para no continuar con un corpus incompleto.
destino = RAIZ / "corpus"
necesarios = ["secciones.jsonl", "chunks.jsonl", "xbrl_facts.parquet",
              "indice/corpus.faiss", "indice/chunks_meta.parquet",
              "indice/MANIFEST.md"]

for nombre in necesarios + ["MANIFEST.md"]:
    salida = destino / nombre
    entrada = RAIZ / "dataset" / nombre
    if not salida.is_file() and entrada.is_file():
        salida.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(entrada, salida)

faltan = [nombre for nombre in necesarios if not (destino / nombre).is_file()]
if faltan:
    raise FileNotFoundError(
        "Falta parte del corpus: " + ", ".join(faltan) + ". "
        "Deja los ZIP proporcionados en clase en dataset/ y vuelve a "
        "ejecutar las celdas de esta sección.")

huella = _sha256(destino / "chunks.jsonl")
for nombre in ["MANIFEST.md", "indice/MANIFEST.md"]:
    ruta = destino / nombre
    if ruta.is_file():
        assert huella in ruta.read_text(encoding="utf-8"), (
            f"chunks.jsonl no coincide con {nombre}.")

print("Corpus completo e índice preparados en", destino)

Corpus completo e índice preparados en c:\Users\Natalia\workplace\Taller-B5-T5-NPL\corpus


## 2. Instalar y configurar el modelo

**Código de clase.** Se conservan sus versiones y su identificador de modelo.
Su disponibilidad depende del proveedor; todavía no se ha verificado en este
entorno. Si una instalación o llamada falla, hay que resolverlo antes de
continuar: el mensaje final de una celda no basta para darla por validada.

In [18]:
# Instalación. Una sola celda, versiones fijadas, salida silenciada.
# Tarda alrededor de minuto y medio: mientras corre, leed la celda siguiente.
%pip install -q \
  langchain==1.3.18 langchain-core==1.6.1 langgraph==1.2.11 \
  langchain-openrouter==0.2.8 langchain-huggingface==1.2.2 \
  sentence-transformers==6.0.1 faiss-cpu==1.15.0 rank-bm25==0.2.2
print("Instalación terminada.")

Note: you may need to restart the kernel to use updated packages.
Instalación terminada.


In [19]:
# Claves. Nunca escritas en el notebook: se leen del entorno, y si no están,
# se piden por teclado sin que queden en la salida de la celda.
import os
import getpass


def pedir_clave(nombre: str, donde: str) -> bool:
    """Deja `nombre` en el entorno si se puede. Devuelve si hay clave."""
    if os.environ.get(nombre):
        print(f"{nombre}: ya estaba en el entorno.")
        return True
    try:
        valor = getpass.getpass(f"{nombre} (se saca en {donde}): ").strip()
    except Exception:                      # sin terminal interactiva
        valor = ""
    if valor:
        os.environ[nombre] = valor
        print(f"{nombre}: guardada en el entorno de esta sesión.")
        return True
    print(f"{nombre}: sin clave. Las celdas que llaman al modelo no van a "
          f"funcionar, pero las de datos sí.")
    return False


HAY_CLAVE = pedir_clave("OPENROUTER_API_KEY", "openrouter.ai/keys")

# Una sola clave. La observabilidad de este curso no necesita ninguna más:
# la trayectoria se imprime en el propio notebook (§6).

OPENROUTER_API_KEY: ya estaba en el entorno.


In [20]:
# Un proveedor es una cadena de texto.
#
# `init_chat_model` devuelve el mismo objeto sea cual sea el proveedor: el
# resto del notebook no se entera de cuál hay debajo. Cambiad la cadena y
# todo lo demás sigue igual.
from langchain.chat_models import init_chat_model

MODELO = "openrouter:google/gemini-3.8-flash"
MAX_TOKENS = 1024       # Limite de salida por llamada; evita reservar 65.536 tokens.

modelo = None
if HAY_CLAVE:
    try:
        modelo = init_chat_model(MODELO, temperature=0, max_tokens=MAX_TOKENS)
        print(modelo.invoke("Responde solo con la palabra: listo").text)
    except Exception as e:
        print(f"No se pudo crear el modelo ({type(e).__name__}: {e}).")

# El mismo código contra otros cuatro proveedores. No los ejecutamos: cuestan
# dinero y hacen falta cuatro claves. El punto es que solo cambia la cadena.
#
#   init_chat_model("openrouter:anthropic/claude-opus-5", temperature=0)
#   init_chat_model("anthropic:claude-opus-5",            temperature=0)
#   init_chat_model("google_genai:gemini-3.8-flash",      temperature=0)
#   init_chat_model("openrouter:auto",                    temperature=0)
#
# El último deja que OpenRouter elija modelo. Sirve para enseñar el concepto y
# NO sirve para evaluar: si el modelo cambia entre dos ejecuciones, la
# comparación baseline-contra-final no significa nada.
#
# `temperature=0` va en todo lo que se vaya a evaluar. Con temperatura alta,
# dos ejecuciones de la misma pregunta dan métricas distintas y no sabéis si
# mejorasteis el sistema o tuvisteis suerte.

# --- verificación de §1 --------------------------------------------------
import pathlib
assert pathlib.Path("corpus/chunks.jsonl").is_file(), \
    "El corpus no está montado: repasad la celda de setup."
assert pathlib.Path("corpus/indice/corpus.faiss").is_file(), \
    "Falta el índice FAISS: descomprimid también indice_faiss.zip."
print("§1 listo.")

listo
§1 listo.


## 3. Cargar los datos y las herramientas

**Código de clase.** Las tres herramientas implementadas se conservan literalmente.

In [21]:
# Cuánto ocupa un 10-K. Los tokens vienen precalculados en el corpus: contar
# en vivo tardaría más que la clase entera.
import json
import pandas as pd

secciones = pd.DataFrame(
    json.loads(l) for l in open("corpus/secciones.jsonl", encoding="utf-8")
)

tabla = secciones.pivot_table(
    index=["ticker", "fiscal_year"], columns="item", values="n_tokens"
).astype(int)
tabla["TOTAL"] = tabla.sum(axis=1)
print(tabla.to_string())

print(f"\nCorpus entero: {secciones.n_tokens.sum():,} tokens en "
      f"{len(secciones)} secciones")
print(f"Informe medio: {tabla['TOTAL'].mean():,.0f} tokens")
print(f"Informe mayor: {tabla['TOTAL'].max():,} · menor: "
      f"{tabla['TOTAL'].min():,}")

mayor = secciones.nlargest(1, "n_tokens").iloc[0]
# Ojo con `mayor.item`: en pandas eso es el método Series.item, no la
# columna. Con una columna que se llama 'item' hay que usar corchetes.
print(f"Sección mayor: {mayor['ticker']} FY{mayor['fiscal_year']} "
      f"Item {mayor['item']} con {mayor['n_tokens']:,} tokens")

item                   1A      7    7A      8  TOTAL
ticker fiscal_year                                  
AAPL   2024         11663   3814   612  15999  32088
       2025         11626   4294   612  16358  32890
AMZN   2024         10318   9597  1614  28097  49626
       2025         10516   9034  1547  29103  50200
GOOGL  2024         14727  11947  1877  30380  58931
       2025         14984  10648  1579  31845  59056
META   2024         33573  12621  1128  28501  75823
       2025         34751  12518  1144  32351  80764
MSFT   2024         12650  10295   409  28455  51809
       2025         11793   9510   409  26500  48212
NVDA   2024         18681   8348   635  26909  54573
       2025         19476   7824   638  27209  55147

Corpus entero: 649,119 tokens en 48 secciones
Informe medio: 54,093 tokens
Informe mayor: 80,764 · menor: 32,088
Sección mayor: META FY2025 Item 1A con 34,751 tokens


In [22]:
from langchain.tools import tool

xbrl = pd.read_parquet("corpus/xbrl_facts.parquet")
print(f"{len(xbrl)} hechos XBRL · {xbrl.concept.nunique()} conceptos "
      f"distintos · {xbrl.ticker.nunique()} compañías")


@tool
def get_xbrl_fact(ticker: str, fiscal_year: int, concept: str) -> str:
    """Devuelve el valor EXACTO de una magnitud financiera tal y como la
    compañía la reportó en XBRL.

    Es la fuente autorizada para cualquier cifra. Úsala SIEMPRE en lugar de
    leer un número del texto del informe.

    Args:
        ticker: Símbolo bursátil, p. ej. 'NVDA'.
        fiscal_year: Ejercicio fiscal reportado, p. ej. 2024.
        concept: Concepto en taxonomía US-GAAP, p. ej. 'Revenues',
            'NetIncomeLoss', 'Assets', 'OperatingIncomeLoss'.

    Devuelve el valor con su unidad y fecha de cierre, o un aviso explícito
    si la compañía no reportó ese concepto en ese ejercicio.
    """
    filas = xbrl[(xbrl.ticker == ticker)
                 & (xbrl.fiscal_year == int(fiscal_year))
                 & (xbrl.concept == concept)]
    if filas.empty:
        disponibles = sorted(
            xbrl[(xbrl.ticker == ticker)
                 & (xbrl.fiscal_year == int(fiscal_year))].concept.unique()
        )
        if not disponibles:
            return (f"No hay datos de {ticker} para FY{fiscal_year} en el "
                    f"corpus. Usa list_available para ver qué hay.")
        return (f"{ticker} no reportó '{concept}' en FY{fiscal_year}. "
                f"Conceptos disponibles: {', '.join(disponibles)}")
    f = filas.iloc[0]
    return (f"{ticker} FY{fiscal_year} · {concept} = {f.value:,.0f} {f.unit} "
            f"(cierre de ejercicio {f.period_end}, según el {f.form})")


# Dos llamadas directas, sin modelo de por medio, para ver qué devuelve.
print(get_xbrl_fact.invoke(
    {"ticker": "NVDA", "fiscal_year": 2024, "concept": "Revenues"}))
print(get_xbrl_fact.invoke(
    {"ticker": "AMZN", "fiscal_year": 2025, "concept": "GrossProfit"}))

# La segunda no es un fallo del corpus: Amazon no etiqueta GrossProfit en
# us-gaap. Que la herramienta lo diga en vez de devolver vacío es la
# diferencia entre un agente que contesta "no está" y uno que se lo inventa.

135 hechos XBRL · 13 conceptos distintos · 6 compañías
NVDA FY2024 · Revenues = 60,922,000,000 USD (cierre de ejercicio 2024-01-28, según el 10-K)
AMZN no reportó 'GrossProfit' en FY2025. Conceptos disponibles: Assets, CashAndCashEquivalentsAtCarryingValue, EarningsPerShareBasic, EarningsPerShareDiluted, NetCashProvidedByUsedInOperatingActivities, NetIncomeLoss, OperatingIncomeLoss, RevenueFromContractWithCustomerExcludingAssessedTax, StockholdersEquity


In [23]:
# La búsqueda en el texto de los informes.
#
# El cuerpo está en miax_s1.py y hoy es CAJA NEGRA a propósito: dentro hay
# troceado, embeddings, un índice FAISS y una decisión de top-k, y cada una de
# esas cuatro cosas se puede hacer mejor o peor. El día 17 se abre la caja, se
# mide lo que hace y se arregla tres veces.
#
# Hoy interesa otra cosa: que es la herramienta CARA y DIFUSA, la contraria de
# get_xbrl_fact. Devuelve texto que se parece a lo que pedisteis, no la
# respuesta.
import miax_s1


@tool
def search_filings(query: str, ticker: str | None = None,
                   fiscal_year: int | None = None,
                   item: str | None = None, k: int = 5) -> str:
    """Busca fragmentos de texto relevantes en los informes 10-K del corpus.

    Úsala para preguntas cualitativas: riesgos, estrategia, litigios,
    comentarios de la dirección. NO la uses para obtener cifras: para eso
    está get_xbrl_fact.

    Args:
        query: Qué buscar, en lenguaje natural.
        ticker: Filtra por compañía si la pregunta la menciona.
        fiscal_year: Filtra por ejercicio si la pregunta lo menciona.
        item: Filtra por sección: '1A' riesgos, '7' MD&A,
            '7A' riesgo de mercado, '8' estados financieros.
        k: Número de fragmentos a devolver.

    Devuelve k fragmentos, cada uno con su chunk_id para poder citarlo.
    """
    return miax_s1.formatear_fragmentos(
        miax_s1.buscar(query, ticker=ticker, fiscal_year=fiscal_year,
                       item=item, k=k)
    )


# La primera llamada tarda unos segundos: carga el modelo de embeddings.
salida = search_filings.invoke({
    "query": "risks from misuse of our AI systems by third parties",
    "ticker": "MSFT", "fiscal_year": 2025, "item": "1A", "k": 3,
})
print(salida[:1200], "...")

# Dos cosas que mirar en esa salida:
#
# El corpus está EN INGLÉS. La consulta también tiene que ir en inglés aunque
# la pregunta del usuario venga en español. Que el agente traduzca la consulta
# es parte de su trabajo.
#
# Y el fragmento empieza a mitad de frase ("of operations."). Nadie ha decidido
# que empiece ahí: es donde cayó el corte del troceador. Eso es exactamente lo
# que se abre y se arregla el día 17.

[MSFT-2025-1A-0017] MSFT FY2025 Item 1A (similitud 0.842)
of operations.



Issues in the development, deployment, and use of AI may result in reputational or competitive harm or liability. We are building AI into many of our offerings, including our productivity services, and we are also making AI available for our customers to use in solutions that they build. This AI may be developed by Microsoft or others, including our strategic partner, OpenAI. We expect these elements of our business to grow. We envision a future in which AI operating in devices, applications, and the cloud helps our customers be more productive in their work and personal lives. As with many innovations, AI presents risks and challenges that could affect its adoption, and therefore our business. AI algorithms or training methodologies may be flawed. Datasets may be overbroad, insufficient, or contain biased or inaccurate information. Content generated by AI systems may be offensive, illegal, inaccurate, or other

In [24]:
# La vía de contexto largo. Existe para que el agente pueda elegir pagarla.
@tool
def read_section(ticker: str, fiscal_year: int, item: str) -> str:
    """Devuelve el TEXTO COMPLETO de una sección de un 10-K.

    Es una herramienta CARA: puede devolver decenas de miles de tokens.
    Úsala solo cuando search_filings devuelva fragmentos insuficientes y
    necesites el contexto entero de una sección concreta.

    Args:
        ticker: Símbolo bursátil, p. ej. 'META'.
        fiscal_year: Ejercicio fiscal, p. ej. 2025.
        item: '1A' riesgos, '7' MD&A, '7A' riesgo de mercado,
            '8' estados financieros.
    """
    filas = secciones[(secciones.ticker == ticker)
                      & (secciones.fiscal_year == int(fiscal_year))
                      & (secciones.item == item)]
    if filas.empty:
        return (f"No hay Item {item} de {ticker} FY{fiscal_year} en el "
                f"corpus. Usa list_available para ver qué hay.")
    f = filas.iloc[0]
    return f.texto


# No la llamamos con el modelo delante: la sección más larga del corpus son
# 34.751 tokens y pagarlos para ver que funciona no tiene sentido. Miramos
# el tamaño de lo que devolvería.
for t, fy, it in [("AAPL", 2024, "7A"), ("META", 2025, "1A")]:
    texto = read_section.invoke(
        {"ticker": t, "fiscal_year": fy, "item": it})
    n = int(secciones[(secciones.ticker == t)
                      & (secciones.fiscal_year == fy)
                      & (secciones.item == it)].iloc[0].n_tokens)
    print(f"{t} FY{fy} Item {it}: {len(texto):,} caracteres, {n:,} tokens")

# Nada impide al agente llamar a read_section cuatro veces seguidas y
# quemarse el presupuesto del grupo en una pregunta. Hoy no hay nada que se
# lo impida; el día 17 se le pone un ToolCallLimitMiddleware.

AAPL FY2024 Item 7A: 3,043 caracteres, 612 tokens
META FY2025 Item 1A: 195,308 caracteres, 34,751 tokens


**Ejercicio completado.** En `list_available()` solo se sustituye el cuerpo pendiente; se conserva su docstring.

In [25]:
# EJERCICIO 1 (10 min) · list_available()
#
# Sin esta herramienta el agente se inventa compañías y ejercicios que no
# están en el corpus: no tiene forma de comprobar el mundo, así que rellena
# el hueco con lo que le suena.
#
# Tiene que devolver, en un texto que el modelo pueda leer: qué tickers hay,
# el nombre de cada compañía, qué ejercicios y qué items. Todo sale de
# `secciones`, que ya está cargado (columnas: ticker, empresa, fiscal_year,
# item, ...).
#
# Dos avisos:
#   - El docstring es parte del ejercicio. Escribid uno que le diga al modelo
#     CUÁNDO llamarla, no solo qué hace.
#   - Devolved un texto, no un DataFrame: lo que devuelve la herramienta se
#     le pasa al modelo tal cual.
@tool
def list_available() -> str:
    """Lista qué compañías, ejercicios y secciones existen en el corpus.

    Úsala SIEMPRE antes de responder que un dato no existe, y antes de
    llamar a cualquier otra herramienta si no estás seguro de que la
    compañía o el ejercicio que te piden estén en el corpus.
    """
    # TODO (alumno): construir el texto a partir de `secciones`
    partes = []
    for (ticker, empresa), filas in secciones.groupby(["ticker", "empresa"]):
        partes.append(f"{ticker} · {empresa}")
        for ejercicio, grupo in filas.groupby("fiscal_year"):
            items = ", ".join(sorted(grupo["item"].astype(str).unique()))
            partes.append(f"  FY{int(ejercicio)}: Items {items}")
    return "\n".join(partes) if partes else "No hay secciones en el corpus."


print(list_available.invoke({}))

AAPL · Apple Inc.
  FY2024: Items 1A, 7, 7A, 8
  FY2025: Items 1A, 7, 7A, 8
AMZN · AMAZON COM INC
  FY2024: Items 1A, 7, 7A, 8
  FY2025: Items 1A, 7, 7A, 8
GOOGL · Alphabet Inc.
  FY2024: Items 1A, 7, 7A, 8
  FY2025: Items 1A, 7, 7A, 8
META · Meta Platforms, Inc.
  FY2024: Items 1A, 7, 7A, 8
  FY2025: Items 1A, 7, 7A, 8
MSFT · MICROSOFT CORP
  FY2024: Items 1A, 7, 7A, 8
  FY2025: Items 1A, 7, 7A, 8
NVDA · NVIDIA CORP
  FY2024: Items 1A, 7, 7A, 8
  FY2025: Items 1A, 7, 7A, 8


In [26]:
# Verificación sin llamar al modelo.
assert "NVDA" in list_available.invoke({})
assert "60,922,000,000" in get_xbrl_fact.invoke(
    {"ticker": "NVDA", "fiscal_year": 2024, "concept": "Revenues"})
assert "no reportó" in get_xbrl_fact.invoke(
    {"ticker": "AMZN", "fiscal_year": 2025, "concept": "GrossProfit"})
print("Las herramientas de datos pasan las comprobaciones iniciales.")

Las herramientas de datos pasan las comprobaciones iniciales.


## 4. El bucle y el agente

**Código de clase y ejercicio completado.** Se conserva el bucle exterior y se completa únicamente la ejecución de herramientas y su `ToolMessage`.

In [27]:
# Qué devuelve exactamente el modelo cuando pide una herramienta.
HERRAMIENTAS = [list_available, get_xbrl_fact, search_filings, read_section]
POR_NOMBRE = {t.name: t for t in HERRAMIENTAS}

if modelo is not None:
    modelo_con_tools = modelo.bind_tools(HERRAMIENTAS)
    respuesta = modelo_con_tools.invoke([
        {"role": "user",
         "content": "¿Cuáles fueron los ingresos de NVIDIA en FY2025?"},
    ])

    print("texto de la respuesta:", repr(respuesta.text))
    print("\ntool_calls, en crudo:")
    for tc in respuesta.tool_calls:
        print(f"  name : {tc['name']}")
        print(f"  args : {tc['args']}")
        print(f"  id   : {tc['id']}")
else:
    modelo_con_tools = None
    print("Sin clave: esta celda necesita el modelo.")

# El texto viene vacío o casi. El contenido de la respuesta ES la petición de
# herramienta. Y el `id` no es decorativo: es lo que empareja cada resultado
# con su petición cuando hay varias a la vez.

texto de la respuesta: ''

tool_calls, en crudo:
  name : get_xbrl_fact
  args : {'concept': 'Revenues', 'fiscal_year': 2025, 'ticker': 'NVDA'}
  id   : call_211748


In [28]:
# EL BUCLE. Treinta líneas, sin framework. El andamiaje está puesto; el
# cuerpo del bucle interior es vuestro.
from langchain.messages import ToolMessage

SYSTEM = """Eres un analista financiero que responde preguntas sobre informes
10-K usando ÚNICAMENTE las herramientas disponibles.

Reglas:
- Para cualquier CIFRA, usa get_xbrl_fact. Nunca leas un número de la prosa.
- Para riesgos, estrategia o comentarios de la dirección, usa search_filings.
- Si no sabes si una compañía o un ejercicio están en el corpus, empieza por
  list_available.
- El corpus está en inglés: escribe las consultas de búsqueda en inglés.
- Cita el chunk_id del fragmento en el que te apoyes.
- Si el dato no está en el corpus, dilo. No lo estimes.
"""


def agente_manual(pregunta: str, max_vueltas: int = 6,
                  verboso: bool = True) -> str:
    mensajes = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": pregunta}]
    for vuelta in range(max_vueltas):
        respuesta = modelo_con_tools.invoke(mensajes)
        mensajes.append(respuesta)

        if not respuesta.tool_calls:
            return respuesta.text

        for tc in respuesta.tool_calls:
            # TODO (alumno): ejecutar la herramienta y añadir el ToolMessage
            #
            #   1. `tc` es un dict con 'name', 'args' e 'id'.
            #   2. POR_NOMBRE[tc["name"]] os da el objeto herramienta;
            #      se ejecuta con .invoke(tc["args"]).
            #   3. El resultado se añade a `mensajes` como
            #      ToolMessage(content=..., tool_call_id=tc["id"],
            #                  name=tc["name"]).
            #      El tool_call_id tiene que ser el de ESTA petición: es lo
            #      que empareja cada resultado con su pregunta cuando el
            #      modelo pide varias herramientas a la vez.
            #   4. content tiene que ser texto: str(resultado).
            #   5. Si la herramienta lanza una excepción, devolvedle el error
            #      al modelo en vez de dejar que rompa el bucle. Un agente que
            #      revienta con la primera excepción no llega a la sesión 2.
            #
            # Si `verboso`, imprimid la vuelta, la herramienta y sus
            # argumentos: sin eso no vais a poder depurar nada.
            if verboso:
                print(f"{vuelta + 1}. {tc['name']}({tc['args']})")
            try:
                resultado = POR_NOMBRE[tc["name"]].invoke(tc["args"])
            except Exception as e:
                resultado = f"Error en {tc['name']}: {type(e).__name__}: {e}"
            mensajes.append(ToolMessage(
                content=str(resultado), tool_call_id=tc["id"],
                name=tc["name"],
            ))
    return "Se agotaron las vueltas sin llegar a una respuesta."


print("agente_manual definido.")

agente_manual definido.


In [29]:
# El esquema de respuesta. Es el contrato §7 del enunciado, literal.
#
# Fijaos en lo que hace: convierte la cita de una súplica en el prompt
# ("por favor, cita la fuente") en un requisito estructural. El modelo no
# puede devolver una respuesta sin decir de dónde sale, porque el esquema no
# valida. Y los evaluadores del día 17 leen campos en lugar de parsear prosa.
from typing import Literal

from pydantic import BaseModel, Field


class RespuestaFinanciera(BaseModel):
    """Respuesta trazable a una pregunta sobre informes 10-K."""

    respuesta: str = Field(
        description="Respuesta en prosa, breve y directa")
    cifra: float | None = Field(
        default=None, description="Valor numérico, si la pregunta pide uno")
    unidad: str | None = Field(
        default=None, description="USD, shares, porcentaje…")
    ticker: str | None = None
    ejercicio: int | None = None
    fuente: Literal["xbrl", "texto", "ambas", "ninguna"] = Field(
        description="De dónde sale el dato. 'ninguna' si no está en el corpus")
    cita: str | None = Field(
        default=None,
        description="Texto literal del informe que respalda la respuesta")
    chunk_id: str | None = Field(
        default=None,
        description="Identificador del fragmento citado, para verificar")


print(json.dumps(RespuestaFinanciera.model_json_schema()["properties"],
                 indent=2, ensure_ascii=False)[:600], "...")

{
  "respuesta": {
    "description": "Respuesta en prosa, breve y directa",
    "title": "Respuesta",
    "type": "string"
  },
  "cifra": {
    "anyOf": [
      {
        "type": "number"
      },
      {
        "type": "null"
      }
    ],
    "default": null,
    "description": "Valor numérico, si la pregunta pide uno",
    "title": "Cifra"
  },
  "unidad": {
    "anyOf": [
      {
        "type": "string"
      },
      {
        "type": "null"
      }
    ],
    "default": null,
    "description": "USD, shares, porcentaje…",
    "title": "Unidad"
  },
  "ticker": {
    "anyOf": [
      ...


In [30]:
# El mismo agente, en cinco líneas.
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agente = None
if HAY_CLAVE:
    agente = create_agent(
        model=modelo,    # Reutiliza temperature y max_tokens ya configurados.
        tools=HERRAMIENTAS,
        system_prompt=SYSTEM,
        response_format=RespuestaFinanciera,
        checkpointer=InMemorySaver(),
    )
    print("Agente montado con", len(HERRAMIENTAS), "herramientas.")
else:
    print("Sin clave: no se puede montar el agente.")

# Qué hace cada línea que vosotros hicisteis a mano:
#
#   tools=            los bind_tools y el diccionario POR_NOMBRE
#   response_format=  la validación de la salida, que no teníais
#   checkpointer=     la memoria entre invocaciones, que tampoco teníais
#   (el bucle)        las treinta líneas de la celda 23
#
# Y trae cosas que no habíais escrito: reintentos, streaming, callbacks para
# instrumentar la ejecución, e interrupciones para humano en el medio, que es
# de lo que va el día 17.
#
# Si el modelo que elijáis no soporta salida estructurada nativa, esa línea
# falla. La salida es envolverlo:
#     from langchain.agents.structured_output import ToolStrategy
#     response_format=ToolStrategy(schema=RespuestaFinanciera)

Agente montado con 4 herramientas.


In [31]:
# Código añadido. La demostración de memoria de clase no se ejecuta aquí.
# pretty_trace se conserva entera; esta variable evita ejecutar su ejemplo.
r2 = None

In [32]:
# Ver la trayectoria.
#
# Una respuesta no se puede juzgar sin ver el camino. Esta función imprime la
# trayectoria: qué herramientas se llamaron, en qué orden, con qué argumentos
# y qué devolvió cada una.
#
# Es también la pieza que hace posible el evaluador `uso_la_tool_correcta` del
# día 17: sin trayectoria no se puede distinguir una respuesta correcta de una
# respuesta correcta por casualidad.
def pretty_trace(resultado) -> None:
    """Imprime la trayectoria: qué herramientas se llamaron, con qué
    argumentos y qué devolvieron."""
    pendientes = {}
    n = 0
    for mensaje in resultado["messages"]:
        for tc in getattr(mensaje, "tool_calls", None) or []:
            n += 1
            pendientes[tc["id"]] = n
            print(f"{n}. {tc['name']}({tc['args']})")
        id_llamada = getattr(mensaje, "tool_call_id", None)
        if id_llamada in pendientes:
            texto = str(mensaje.content).replace("\n", " ")
            print(f"   -> {texto[:160]}"
                  f"{'...' if len(texto) > 160 else ''}")
    print(f"\n{n} llamadas a herramienta")
    e = resultado.get("structured_response")
    if e is not None:
        print(f"respuesta: {e.respuesta}")
        print(f"fuente: {e.fuente} · cifra: {e.cifra} {e.unidad or ''} "
              f"· cita: {e.chunk_id}")


if r2 is not None:
    pretty_trace(r2)

## 5. Responder y conservar la ejecución

**Código añadido.** `responder(pregunta)` devuelve una `RespuestaFinanciera`.
Cada pregunta usa una conversación nueva, para que una respuesta anterior no
contamine la siguiente. La evaluación utiliza el mismo camino de ejecución
y guarda también la trayectoria, los tokens y la latencia.

El límite de recursión es un tope de pasos del framework, no un límite de
llamadas a herramientas. El middleware específico se añadirá en la sesión 2.
Para calcular costes, rellenad los dos precios por millón de tokens de
vuestro proveedor. Hasta entonces el coste se guarda como `null`, nunca cero.

In [33]:
import time
import uuid

LIMITE_RECURSION = 30
PRECIO_ENTRADA = None       # USD por millón de tokens del modelo utilizado
PRECIO_SALIDA = None        # USD por millón de tokens del modelo utilizado


def ejecutar(pregunta: str) -> dict:
    """Una pregunta, una conversación y las medidas de esa ejecución."""
    if agente is None:
        raise RuntimeError("Configura la clave y crea el agente antes de responder.")
    if not isinstance(pregunta, str) or not pregunta.strip():
        raise ValueError("La pregunta tiene que ser un texto no vacío.")
    inicio = time.perf_counter()
    resultado = agente.invoke(
        {"messages": [{"role": "user", "content": pregunta}]},
        config={"configurable": {"thread_id": str(uuid.uuid4())},
                "recursion_limit": LIMITE_RECURSION},
    )
    latencia = time.perf_counter() - inicio
    respuesta = resultado["structured_response"]
    assert isinstance(respuesta, RespuestaFinanciera)
    mensajes = resultado["messages"]
    llamadas = [tc for m in mensajes
                for tc in (getattr(m, "tool_calls", None) or [])
                if tc["name"] in POR_NOMBRE]
    salidas = [m.model_dump(mode="json") for m in mensajes
               if getattr(m, "type", None) == "tool"]
    usos = [getattr(m, "usage_metadata", None) for m in mensajes
            if getattr(m, "type", None) == "ai"]
    completo = bool(usos) and all(
        u is not None and "input_tokens" in u and "output_tokens" in u
        for u in usos)
    entrada = sum(u["input_tokens"] for u in usos) if completo else None
    salida = sum(u["output_tokens"] for u in usos) if completo else None
    coste = None
    if completo and PRECIO_ENTRADA is not None and PRECIO_SALIDA is not None:
        coste = (entrada * PRECIO_ENTRADA + salida * PRECIO_SALIDA) / 1e6
    return {
        "respuesta": respuesta.model_dump(mode="json"),
        "llamadas": llamadas, "salidas_herramientas": salidas,
        "latencia_s": latencia, "llamadas_herramienta": len(llamadas),
        "tokens_entrada": entrada, "tokens_salida": salida,
        "coste_estimado_usd": coste,
    }


def responder(pregunta: str) -> RespuestaFinanciera:
    """Responde una pregunta con el esquema obligatorio de la práctica."""
    return RespuestaFinanciera(**ejecutar(pregunta)["respuesta"])


# Ejemplo de uso, cuando queráis hacer una llamada:
# respuesta = responder("¿Cuál fue el revenue de NVIDIA en FY2024?")
# print(respuesta.model_dump_json(indent=2))

## 6. Cargar las preguntas y validarlas

**Código añadido y validador de clase.** Se carga el fichero de ejemplo desde
su ubicación original. Tiene tres preguntas y sirve para probar el formato;
no sustituye las 20 preguntas propias. El validador se conserva literalmente.

In [34]:
RUTA_GOLDEN = RAIZ / "Material_Clase/golden_set_ejemplo.jsonl"
golden = [json.loads(linea) for linea in RUTA_GOLDEN.read_text(
    encoding="utf-8").splitlines() if linea.strip()]
print(f"{RUTA_GOLDEN.name}: {len(golden)} preguntas de ejemplo")

golden_set_ejemplo.jsonl: 3 preguntas de ejemplo


In [35]:
# Vuestras 20 preguntas, y el validador que tienen que pasar antes de
# entregarlas. Un golden set que no pasa el validador no se corrige: se
# devuelve.
PLANTILLA = {
    "id": "g3-001",
    "pregunta": "¿Cuál fue el revenue de NVIDIA en el ejercicio 2024?",
    "familia": "numerica",              # extractiva | numerica | comparativa
    "ticker": "NVDA",
    "fiscal_year": 2024,
    "respuesta_esperada": "60.922 millones de dólares",
    "cifra_esperada": 60922000000.0,
    "unidad": "USD",
    "concept_xbrl": "Revenues",
    "item_esperado": None,
    "ancla_texto": None,
    "ancla_inicio": None,
    "ancla_fin": None,
    "chunk_id_esperado": None,
    "herramienta_esperada": ["get_xbrl_fact"],
    "autor": "grupo-3",
}

CAMPOS = set(PLANTILLA)
FAMILIAS = {"extractiva", "numerica", "comparativa"}


def validar(preguntas: list[dict], exigir_20: bool = True) -> list[str]:
    """Los problemas del fichero, uno por línea. Lista vacía = correcto."""
    problemas = []
    tickers = set(secciones.ticker)
    ejercicios = set(secciones.fiscal_year.astype(int))
    vistos = set()

    for p in preguntas:
        pid = p.get("id", "(sin id)")
        if faltan := CAMPOS - set(p):
            problemas.append(f"{pid}: faltan campos {sorted(faltan)}")
            continue
        if p["id"] in vistos:
            problemas.append(f"{pid}: id repetido")
        vistos.add(p["id"])
        if p["familia"] not in FAMILIAS:
            problemas.append(f"{pid}: familia '{p['familia']}' no válida")
        if p["ticker"] not in tickers:
            problemas.append(f"{pid}: {p['ticker']} no está en el corpus")
        if int(p["fiscal_year"]) not in ejercicios:
            problemas.append(f"{pid}: FY{p['fiscal_year']} no está en el "
                             f"corpus")
        if p["familia"] in {"numerica", "comparativa"}:
            if p.get("cifra_esperada") is None:
                problemas.append(f"{pid}: numérica sin cifra_esperada")
            concepto = p.get("concept_xbrl")
            hay = xbrl[(xbrl.ticker == p["ticker"])
                       & (xbrl.fiscal_year == int(p["fiscal_year"]))
                       & (xbrl.concept == concepto)]
            if concepto and hay.empty:
                problemas.append(
                    f"{pid}: {p['ticker']} no reporta '{concepto}' en "
                    f"FY{p['fiscal_year']}. El concepto se mira en "
                    f"xbrl_facts.parquet, nunca por analogía con otra "
                    f"compañía.")
        if p["familia"] in {"extractiva", "comparativa"}:
            ancla = p.get("ancla_texto")
            if not ancla:
                problemas.append(f"{pid}: extractiva sin ancla_texto")
            elif len(ancla.split()) > 40:
                problemas.append(
                    f"{pid}: ancla de {len(ancla.split())} palabras. Una "
                    f"frase. Así no medís vuestro retrieval, medís vuestro "
                    f"tamaño de ventana.")
        if not p.get("herramienta_esperada"):
            problemas.append(f"{pid}: sin herramienta_esperada")

    if exigir_20:
        if len(preguntas) != 20:
            problemas.append(f"hacen falta 20 preguntas, hay {len(preguntas)}")
        n_comp = sum(p.get("familia") == "comparativa" for p in preguntas)
        if n_comp < 6:
            problemas.append(f"hacen falta 6 comparativas, hay {n_comp}")
    return problemas


problemas = validar(golden, exigir_20=False)
print("Validando el fichero cargado:")
print("\n".join(f"  - {p}" for p in problemas) or "  sin problemas")

# --- verificación de §7 --------------------------------------------------
assert not validar([PLANTILLA], exigir_20=False), \
    "La plantilla debería pasar su propio validador."
assert validar([{**PLANTILLA, "ticker": "TSLA"}], exigir_20=False), \
    "El validador tiene que rechazar una compañía que no está en el corpus."
print("\n§7 listo. El validador funciona; ahora escribid las preguntas.")

Validando el fichero cargado:
  sin problemas

§7 listo. El validador funciona; ahora escribid las preguntas.


## 7. Evaluación inicial del baseline

**Código añadido.** `evaluar(ruta_jsonl)` ejecuta el fichero y conserva cada
respuesta inmediatamente en una carpeta nueva de `resultados/baseline/`.
Se guardan los errores por pregunta para poder clasificarlos el día 17.

Las comprobaciones iniciales son deliberadamente limitadas:

- Cifra y unidad frente al golden set, con tolerancia relativa de `1e-6`
  y absoluta de `0.01`, en la unidad declarada, sin convertir unidades.
- Cita literal dentro del fragmento indicado, con la empresa y el ejercicio
  esperados. Esto comprueba trazabilidad, pero no que respalde toda la respuesta.
- Presencia de las herramientas esperadas en la trayectoria. Todavía no se
  juzga si sus argumentos o su orden eran los correctos.
- `recall@k` sobre el ancla literal, por cada pregunta con ancla. Se mide la
  búsqueda directa con la consulta en español del golden set y sus filtros;
  no es la búsqueda reformulada por el agente. Se mantiene fija para comparar.

No se presenta la combinación de estas comprobaciones como un porcentaje
de aciertos: faltan los evaluadores completos, especialmente para comparar
dos ejercicios. El ejemplo comparativo de clase espera la cifra del último
ejercicio aunque pregunta por crecimiento; revisad esa convención al escribir
vuestro golden set. Una respuesta correcta en porcentaje podría no coincidir.

In [36]:
import math
from datetime import datetime, timezone

TOLERANCIA_RELATIVA = 1e-6
TOLERANCIA_ABSOLUTA = 0.01
K_RETRIEVAL = 5

chunks = pd.DataFrame(json.loads(linea) for linea in
                      open("corpus/chunks.jsonl", encoding="utf-8"))
CHUNKS_POR_ID = {fila["chunk_id"]: fila
                 for fila in chunks.to_dict(orient="records")}


def comprobar(pregunta: dict, ejecucion: dict) -> dict:
    """Comprobaciones iniciales. None significa que no aplica."""
    respuesta = ejecucion["respuesta"]
    cifra_correcta = None
    esperada = pregunta.get("cifra_esperada")
    if esperada is not None:
        cifra = respuesta.get("cifra")
        cifra_correcta = (
            cifra is not None
            and respuesta.get("unidad") == pregunta.get("unidad")
            and math.isclose(float(cifra), float(esperada),
                             rel_tol=TOLERANCIA_RELATIVA,
                             abs_tol=TOLERANCIA_ABSOLUTA))

    cita_localizable = None
    if pregunta.get("ancla_texto"):
        fragmento = CHUNKS_POR_ID.get(respuesta.get("chunk_id"))
        cita = respuesta.get("cita")
        cita_localizable = bool(
            fragmento and cita and cita.strip()
            and cita in fragmento["texto"]
            and fragmento["ticker"] == pregunta["ticker"]
            and int(fragmento["fiscal_year"]) == int(pregunta["fiscal_year"]))

    usadas = {tc["name"] for tc in ejecucion["llamadas"]}
    return {
        "cifra_coincide_golden": cifra_correcta,
        "cita_localizable": cita_localizable,
        "herramientas_esperadas_presentes":
            set(pregunta["herramienta_esperada"]).issubset(usadas),
    }


def evaluar(ruta_jsonl) -> pd.DataFrame:
    """Ejecuta un JSONL y guarda respuestas, trazas y métricas del baseline."""
    if agente is None:
        raise RuntimeError("Primero configura la clave y crea el agente.")
    ruta = Path(ruta_jsonl)
    preguntas = [json.loads(linea) for linea in ruta.read_text(
        encoding="utf-8").splitlines() if linea.strip()]
    if not preguntas:
        raise ValueError("El fichero de preguntas está vacío.")
    problemas = validar(preguntas, exigir_20=False)
    if problemas:
        raise ValueError("El golden set no es válido:\n" + "\n".join(problemas))

    fecha = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    destino = RAIZ / "resultados/baseline" / (fecha + "_" + uuid.uuid4().hex[:8])
    destino.mkdir(parents=True, exist_ok=False)
    shutil.copy2(ruta, destino / "preguntas.jsonl")
    # Guardamos el código usado, junto a los resultados de esta versión.
    shutil.copy2(RAIZ / "src/Baseline_Agente_10K.ipynb", destino / "baseline.ipynb")
    shutil.copy2(RAIZ / "Material_Clase/miax_s1.py", destino / "miax_s1.py")
    configuracion = {
        "sistema": "baseline", "modelo": MODELO,
        "fecha_utc": fecha, "limite_recursion": LIMITE_RECURSION,
        "max_tokens": MAX_TOKENS,
        "precio_entrada_usd_millon": PRECIO_ENTRADA,
        "precio_salida_usd_millon": PRECIO_SALIDA,
        "k_retrieval": K_RETRIEVAL,
        "consulta_retrieval": "pregunta original con filtros del golden set",
        "tolerancia_relativa": TOLERANCIA_RELATIVA,
        "tolerancia_absoluta": TOLERANCIA_ABSOLUTA,
        "chunks_sha256": hashlib.sha256(
            (RAIZ / "corpus/chunks.jsonl").read_bytes()).hexdigest(),
    }
    (destino / "configuracion.json").write_text(
        json.dumps(configuracion, ensure_ascii=False, indent=2), encoding="utf-8")

    filas = []
    with open(destino / "respuestas.jsonl", "w", encoding="utf-8") as archivo:
        for pregunta in preguntas:
            fila = {"id": pregunta["id"], "familia": pregunta["familia"],
                    "error": None, "error_retrieval": None,
                    "recall_ancla_at_k": None}
            inicio = time.perf_counter()
            try:
                ejecucion = ejecutar(pregunta["pregunta"])
                fila.update(ejecucion)
                fila.update(comprobar(pregunta, ejecucion))
            except Exception as e:
                fila.update(error=f"{type(e).__name__}: {e}",
                            latencia_s=time.perf_counter() - inicio)
                # En una ejecución fallida el consumo es desconocido, no cero.
            if pregunta.get("ancla_texto"):
                try:
                    fragmentos = miax_s1.buscar(
                        pregunta["pregunta"], ticker=pregunta["ticker"],
                        fiscal_year=pregunta["fiscal_year"],
                        item=pregunta.get("item_esperado"), k=K_RETRIEVAL)
                    fila["recall_ancla_at_k"] = int(any(
                        pregunta["ancla_texto"] in f["texto"] for f in fragmentos))
                    fila["retrieval_chunk_ids"] = [f["chunk_id"] for f in fragmentos]
                except Exception as e:
                    fila["error_retrieval"] = f"{type(e).__name__}: {e}"
            archivo.write(json.dumps(fila, ensure_ascii=False) + "\n")
            archivo.flush()
            filas.append(fila)
            print(pregunta["id"], "ERROR" if fila["error"] else "respondida")

    columnas = ["cifra_coincide_golden", "cita_localizable",
                "herramientas_esperadas_presentes", "recall_ancla_at_k",
                "latencia_s", "llamadas_herramienta", "tokens_entrada",
                "tokens_salida", "coste_estimado_usd"]
    tabla = pd.DataFrame(filas)
    for columna in columnas:
        if columna not in tabla:
            tabla[columna] = float("nan")
    tabla[["id", "familia", "error", "error_retrieval"] + columnas].to_csv(
        destino / "metricas.csv", index=False)
    tabla["fallo_ejecucion"] = tabla["error"].notna()
    # El count deja visible cuántas preguntas sustentan cada media.
    resumen = tabla.groupby("familia")[columnas + ["fallo_ejecucion"]].agg(
        ["mean", "count"])
    resumen.to_csv(destino / "resumen_por_familia.csv")
    print("\nResultados guardados en", destino)
    print(resumen.to_string())
    return tabla

## 8. Ejecutar y guardar el baseline

Primero probad con las tres preguntas de ejemplo. Después guardad vuestras
20 preguntas en `src/golden_set_propio.jsonl`, cambiad la ruta y activad
`EXIGIR_20`. Guardad el notebook antes de evaluar: se archiva su versión en
disco. Cada ejecución genera una carpeta nueva y conserva las anteriores.

El baseline queda medido cuando termina esta evaluación con el modelo real.
Un notebook creado o una comprobación sin API no equivalen a ese resultado.

In [37]:
# Código añadido. Activad la ejecución cuando los datos y la clave estén listos.
EJECUTAR_EVALUACION = False
EXIGIR_20 = False
RUTA_EVALUACION = RAIZ / "Material_Clase/golden_set_ejemplo.jsonl"
# RUTA_EVALUACION = RAIZ / "src/golden_set_propio.jsonl"

if EJECUTAR_EVALUACION:
    preguntas = [json.loads(linea) for linea in RUTA_EVALUACION.read_text(
        encoding="utf-8").splitlines() if linea.strip()]
    problemas = validar(preguntas, exigir_20=EXIGIR_20)
    assert not problemas, "\n".join(problemas)
    resultados_baseline = evaluar(RUTA_EVALUACION)
else:
    print("Preparado. Activa EJECUTAR_EVALUACION para llamar al modelo.")

Preparado. Activa EJECUTAR_EVALUACION para llamar al modelo.
